### Schwierigkeiten [Bwinf43 2024](https://bwinf.de/bundeswettbewerb/43/) - A2

[Video]()

<img src='bild1.png' width='251'>
<img src='bild2.png' width='251'>


#### Beispieldaten

In [30]:
# Anschauen der Beispieldaten 0-5
nr = 2
print(f'Beispiel {nr}:')
f = open(f'beispieldaten/schwierigkeiten{nr}.txt')
print(f.read().strip())
f.close()

Beispiel 2:
6 8 6
A < B < C
E < C < D
E < C < A
E < F < G < H
H < F
D < E
A B D E F G


Jede Datei ist so aufgebaut:

- Erste Zeile: 3 ganze positive Zahlen n, m und k.
	n: Anzahl der Klausuren.
	m: Gesamtanzahl an Aufgaben.
	k: Anzahl an Aufgaben für welche eine gute Anordnung gefunden werden soll.
- Folgende n Zeilen: Abstufungen der alten Klausuren. Jede Zeile beschreibt eine Klausur und hat die Form $a_1$ < $a_2$ < ..., wobei jedes $a_i$ eine Aufgabe beschreibt. Eine Klausur hat mindestens 2 und maximal m Aufgaben.
- Die Aufgaben werden durch die ersten m Buchstaben des Alphabets von A, B, ... bezeichnet.
- Die letzte Zeile enthält die k Aufgaben, für die eine gute Anordnung gefunden werden soll.

Die Datei schwierigkeiten0.txt entspricht dem Beispiel aus der Aufgabenstellung.

Hier eine Beispieleingabe:

    2 5 3
    B < A < D < E
    C < D < E
    A E C

In diesem Beispiel gibt es zwei alte Klausuren mit fünf verschiedenen Aufgaben: A, B, C, D und E. Es soll für drei der Aufgaben eine neue Anordnung gefunden werden. In der 2. Zeile steht die Schwierigkeitsabstufung der ersten Altklausur. B ist leichter als A ist leichter als D ist leichter als E. In der 3. Zeile die Abstufung der 2. Altklausur. Hier ist C leichter als D und D leichter als E. Die letzte Zeile gibt an, dass für die Aufgaben A, E und C eine neue Anordnung gefunden werden soll.

#### Einlesen der Daten

In [54]:
nr = 5
print(f'Beispiel {nr}:')
f = open(f'beispieldaten/schwierigkeiten{nr}.txt')
anzahl_klausuren, gesamtzahl_aufgaben, anzahl_aufgaben = [int(x) for x in f.readline().split()]
klausuren = []
for i in range(anzahl_klausuren):
    a = f.readline().replace(' ','').strip().split('<')    # Liste mit den Abstufungen 
    klausuren.append(a)
aufgaben = [x for x in f.readline().split()]
f.close()
print(f'{anzahl_klausuren=}, {gesamtzahl_aufgaben=}, {anzahl_aufgaben=}')
print(f'Reihenfolge in alten Klausuren:')
for kl in klausuren:
    print(kl)
print(f'Aktuelle Aufgaben: {aufgaben}')

Beispiel 5:
anzahl_klausuren=11, gesamtzahl_aufgaben=26, anzahl_aufgaben=26
Reihenfolge in alten Klausuren:
['H', 'S', 'C', 'A', 'G']
['S', 'O', 'J', 'L', 'F']
['O', 'M', 'X', 'D', 'U']
['C', 'S', 'E', 'N', 'M']
['E', 'M', 'F', 'X', 'B']
['D', 'G', 'P', 'X', 'A']
['R', 'L', 'X', 'U', 'T']
['M', 'O', 'F', 'V', 'D']
['S', 'O', 'U', 'T', 'P']
['Z', 'Q', 'K', 'X', 'I']
['B', 'W', 'I', 'Y']
Aktuelle Aufgaben: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


Graph erstellen

In [55]:
V = {chr(ord('A') + i) for i in range(gesamtzahl_aufgaben)}     # Knoten
G = {v:set() for v in V}

for kl in klausuren:
    for i in range(len(kl)-1):
        G[kl[i]].add(kl[i+1])

G

{'F': {'V', 'X'},
 'D': {'G', 'U'},
 'U': {'T'},
 'P': {'X'},
 'C': {'A', 'S'},
 'W': {'I'},
 'M': {'F', 'O', 'X'},
 'S': {'C', 'E', 'O'},
 'Z': {'Q'},
 'O': {'F', 'J', 'M', 'U'},
 'B': {'W'},
 'Y': set(),
 'A': {'G'},
 'T': {'P'},
 'N': {'M'},
 'G': {'P'},
 'J': {'L'},
 'I': {'Y'},
 'E': {'M', 'N'},
 'Q': {'K'},
 'K': {'X'},
 'L': {'F', 'X'},
 'V': {'D'},
 'H': {'S'},
 'X': {'A', 'B', 'D', 'I', 'U'},
 'R': {'L'}}

Wir bestimmen die strongly connected components (scc) im Graphen. Alle Elemente einer Komponente erachten wir als gleich schwierig. Sie können also in beliebiger Reihenfolge erscheinen. Die Komponenten selbst werden topologisch sortiert, so beachten wir bei der Aufzählung die *schwierig*-Beziehung.

#### SCCs berechnen

Eine Erklärung mit Video zu dem Algorithmus findet sich [hier](https://github.com/ktheu/InfoKurs/blob/gh-pages/Flipped/scc/scc.ipynb)

In [56]:
# explore reverse Graph, merke postvisit nr
def exploreR(v):
    global counter
    visited[v] = True
    for w in Gr[v]:
        if not visited[w]:
            exploreR(w)
    postvisit[v] = counter
    counter+=1

# explore Graph, merke connected component number
def explore(v):
    visited[v] = True
    ccnum[v] = cc
    for w in G[v]:
        if not visited[w]:
            explore(w)

# Gr ist der reverse Graph von G
Gr = {v:set() for v in G}
for v in G:
    for w in G[v]:
        Gr[w].add(v)

# dfs durch den reversen Graphen, merke postvisit-nr
visited = {v : False for v in Gr}
postvisit = {v : 0 for v in Gr}
counter = 1
for v in Gr:
    if not visited[v] :
        exploreR(v)

# die Knoten in der umgekehrten postvisit-Reihenfolge des reversen Graphen
vlist = sorted(Gr,key=lambda v: postvisit[v],reverse=True)

# dfs durch den Ausgangsgraphen, in der vlist Reihenfolge, dabei component nr merken
visited = {v : False for v in G}
ccnum = {v : 0 for v in G}
cc = 1
for v in vlist:
    if not visited[v] :
        explore(v)
        cc+=1

print('Die aktuellen Aufgaben:')
for i in range(1,cc):
    result = [v for v in G if ccnum[v] == i if v in aufgaben]
    print(i,'-',*result)

Die aktuellen Aufgaben:
1 - Y
2 - I
3 - W
4 - B
5 - D U P A T G X
6 - V
7 - K
8 - Q
9 - Z
10 - F
11 - L
12 - R
13 - J
14 - M O
15 - N
16 - E
17 - C S
18 - H


#### Das gesamte Programm

In [23]:
nr = 0
print(f'Beispiel {nr}:')
f = open(f'beispieldaten/schwierigkeiten{nr}.txt')
anzahl_klausuren, gesamtzahl_aufgaben, anzahl_aufgaben = [int(x) for x in f.readline().split()]
klausuren = []
for i in range(anzahl_klausuren):
    a = f.readline().replace(' ','').strip().split('<')    # Liste mit den Abstufungen 
    klausuren.append(a)
aufgaben = [x for x in f.readline().split()]
f.close()

V = {chr(ord('A') + i) for i in range(gesamtzahl_aufgaben)}     # Knoten
G = {v:set() for v in V}

for kl in klausuren:
    for i in range(len(kl)-1):
        G[kl[i]].add(kl[i+1])

# explore reverse Graph, merke postvisit nr
def exploreR(v):
    global counter
    visited[v] = True
    for w in Gr[v]:
        if not visited[w]:
            exploreR(w)
    postvisit[v] = counter
    counter+=1

# explore Graph, merke connected component number
def explore(v):
    visited[v] = True
    ccnum[v] = cc
    for w in G[v]:
        if not visited[w]:
            explore(w)

# Gr ist der reverse Graph von G
Gr = {v:set() for v in G}
for v in G:
    for w in G[v]:
        Gr[w].add(v)

# dfs durch den reversen Graphen, merke postvisit-nr
visited = {v : False for v in Gr}
postvisit = {v : 0 for v in Gr}
counter = 1
for v in Gr:
    if not visited[v] :
        exploreR(v)

# die Knoten in der umgekehrten postvisit-Reihenfolge des reversen Graphen
vlist = sorted(Gr,key=lambda v: postvisit[v],reverse=True)

# dfs durch den Ausgangsgraphen, in der vlist Reihenfolge, dabei component nr merken
visited = {v : False for v in G}
ccnum = {v : 0 for v in G}
cc = 1
for v in vlist:
    if not visited[v] :
        explore(v)
        cc+=1

results = []
for i in range(1,cc):
    for w in [v for v in G if ccnum[v] == i if v in aufgaben]:
        results.append(w)
results.reverse()
print(results)

Beispiel 0:
['B', 'E', 'D', 'F', 'C']
